In [8]:
import pandas as pd
import numpy as np

# 1. Carrega os metadados (assumindo que 'metadata.csv' ou 'METADADOS_ATUALIZADO' tem Sector, Industry e mcap_YEAR)
temp = pd.read_csv("../../data/02_clean/metadata - metadata_att (2).csv") 

# 2. Carrega as métricas que já estão no formato longo (node, year, hrm, pozzi, beta, momentum)
df_metrics = pd.read_parquet("../../data/02_clean/df_metrics.parquet")
df_metrics["hrm"] = -df_metrics["hrm"]

# 3. Cria os decis dinamicamente ano a ano para evitar vazamento de dados
decile_labels = [f"decil_{i}" for i in range(1, 11)]

def create_deciles(series):
    return pd.qcut(series, q=10, labels=decile_labels, duplicates='drop')

df_metrics['decil_hrm'] = df_metrics.groupby('year')['hrm'].transform(create_deciles)
df_metrics['decil_pozzi'] = df_metrics.groupby('year')['pozzi'].transform(create_deciles)

years = range(2014, 2025)
metrics_to_run = ["hrm", "pozzi"]

for metric in metrics_to_run:
    all_years_df = []
    
    for year in years:
        print(f"Processing year {year}, metric={metric}")
        
        # Filtra os dados apenas para o ano em loop
        df_year = df_metrics[df_metrics['year'] == f'{year}'].copy()
        
        # Renomeia 'node' para 'Ticker' para conseguir dar merge
        df_year = df_year.rename(columns={'node': 'Ticker'})
        
        # Define o nome do portfolio como sendo o decil que o ativo caiu
        df_year['portfolio'] = df_year[f'decil_{metric}']
        
        # Cruza com temp para buscar Sector, Industry e mcap_20XX
        final_df = df_year.merge(temp, on="Ticker", how="inner")
        
        # Cria as colunas com sufixo do ano (para reproduzir seu DataFrame esparso)
        final_df[f'beta_{year}'] = final_df['beta']
        final_df[f'momentum_{year}'] = final_df['momentum']
        
        # Filtra e limpa as colunas. Removemos a parte do 'cut' daqui.
        cols_to_keep = [
            "Ticker", "Sector", "Industry", "portfolio", "year", 
            f"mcap_{year}", f"beta_{year}", f"momentum_{year}"
        ]
        
        # Se 'temp' já tinha problemas de vírgulas no mcap, você pode incluir o replace aqui, 
        # caso contrário o merge acima já traz certinho.
        if f'mcap_{year}' in final_df.columns:
            # (Opcional) Limpa vírgula se ainda for string
            if final_df[f'mcap_{year}'].dtype == object:
                final_df[f'mcap_{year}'] = final_df[f'mcap_{year}'].astype(str).str.replace(",", ".", regex=False).apply(pd.to_numeric, errors="coerce")
        
        # Mantém só as colunas que de fato existem após o merge
        cols_to_keep = [c for c in cols_to_keep if c in final_df.columns]
        
        final_df = final_df[cols_to_keep]
        all_years_df.append(final_df)
        
    # Concatena os anos. Linhas de 2015 terão NaN nas colunas de beta_2023, etc.
    final_panel_df = pd.concat(all_years_df, ignore_index=True)
    
    # Salva gerando complete_metadata_hrm.csv e complete_metadata_pozzi.csv
    if metric == "hrm":
        metric = "hcm"
    output_path = f"../../data/07_portfolios_metadata/complete_metadata_{metric}.parquet"
    final_panel_df.to_parquet(output_path, index=False)
    print(f"Salvo: complete_metadata_{metric}.csv\n")

Processing year 2014, metric=hrm
Processing year 2015, metric=hrm
Processing year 2016, metric=hrm
Processing year 2017, metric=hrm
Processing year 2018, metric=hrm
Processing year 2019, metric=hrm
Processing year 2020, metric=hrm
Processing year 2021, metric=hrm
Processing year 2022, metric=hrm
Processing year 2023, metric=hrm
Processing year 2024, metric=hrm
Salvo: complete_metadata_hcm.csv

Processing year 2014, metric=pozzi
Processing year 2015, metric=pozzi
Processing year 2016, metric=pozzi
Processing year 2017, metric=pozzi
Processing year 2018, metric=pozzi
Processing year 2019, metric=pozzi
Processing year 2020, metric=pozzi
Processing year 2021, metric=pozzi
Processing year 2022, metric=pozzi
Processing year 2023, metric=pozzi
Processing year 2024, metric=pozzi
Salvo: complete_metadata_pozzi.csv



In [9]:
metric = "hcm"
output_path = f"../../data/07_portfolios_metadata/complete_metadata_{metric}.parquet"

import pandas as pd 

pd.read_parquet(output_path)

,Ticker,Sector,Industry,portfolio,year,mcap_2014,beta_2014,momentum_2014,mcap_2015,beta_2015,...,momentum_2021,mcap_2022,beta_2022,momentum_2022,mcap_2023,beta_2023,momentum_2023,mcap_2024,beta_2024,momentum_2024
0,A,Healthcare,Diagnostics & Research,decil_6,2014,1.375584e+10,1.289674,0.036170,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AA,Basic Materials,Aluminum,decil_7,2014,6.477024e+09,1.470522,0.518100,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AAL,Industrials,Airlines,decil_8,2014,3.734874e+10,1.605365,1.117107,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AAME,Financial,Insurance - Life,decil_9,2014,8.298173e+07,0.045365,-0.002096,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AAON,Industrials,Building Products & Equipment,decil_1,2014,8.074726e+08,1.682752,0.066075,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40657,ZTS,Healthcare,Drug Manufacturers - Specialty & Generic,decil_5,2024,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.292747e+10,0.623014,-0.166471
40658,ZUMZ,Consumer Cyclical,Apparel Retail,decil_1,2024,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.539357e+08,1.244622,-0.073231
40659,ZVRA,Healthcare,Biotechnology,decil_6,2024,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.857353e+08,NaN,NaN
40660,ZWS,Industrials,Pollution & Treatment Controls,decil_3,2024,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.353906e+09,NaN,NaN


In [27]:
df_metrics[df_metrics['year'] == '2014']

,node,hrm,pozzi,degree,closeness,eig,year,beta,momentum,decil_hrm,decil_pozzi
0,A,0.113502,0.162029,2.219453,0.289924,0.000978,2014,1.289674,0.036170,decil_6,decil_1
1,AA,0.242146,0.222550,1.851429,0.271225,0.000676,2014,1.470522,0.518100,decil_7,decil_9
2,AAL,0.411204,0.213583,2.391266,0.237571,0.000902,2014,1.605365,1.117107,decil_8,decil_8
3,AAME,0.566156,0.212748,0.223170,0.226859,0.000381,2014,0.045365,-0.002096,decil_9,decil_8
4,AAON,-0.892336,0.186629,2.481132,0.331172,0.034447,2014,1.682752,0.066075,decil_1,decil_6
...,...,...,...,...,...,...,...,...,...,...,...
2718,ZNB,0.473366,0.216124,0.299424,0.243640,0.000071,2014,0.628092,-0.102083,decil_9,decil_8
2719,ZROZ,0.382266,0.214811,3.360004,0.238718,0.000283,2014,-0.692103,0.467750,decil_8,decil_8
2720,ZSL,0.429236,0.276925,6.440218,0.211660,0.000012,2014,0.112141,0.300398,decil_8,decil_10
2721,ZTR,0.098267,0.105097,0.509073,0.301451,0.001542,2014,0.451517,0.094877,decil_5,decil_1


In [30]:
final_panel_df

,Ticker,Sector,Industry,portfolio,year,beta_2014,momentum_2014,mcap_2015,beta_2015,momentum_2015,...,momentum_2021,mcap_2022,beta_2022,momentum_2022,mcap_2023,beta_2023,momentum_2023,mcap_2024,beta_2024,momentum_2024
0,A,Healthcare,Diagnostics & Research,decil_1,2014,1.289674,0.036170,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AA,Basic Materials,Aluminum,decil_9,2014,1.470522,0.518100,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AAL,Industrials,Airlines,decil_8,2014,1.605365,1.117107,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AAME,Financial,Insurance - Life,decil_8,2014,0.045365,-0.002096,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AAON,Industrials,Building Products & Equipment,decil_6,2014,1.682752,0.066075,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40657,ZTS,Healthcare,Drug Manufacturers - Specialty & Generic,decil_1,2024,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.292747e+10,0.623014,-0.166471
40658,ZUMZ,Consumer Cyclical,Apparel Retail,decil_6,2024,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.539357e+08,1.244622,-0.073231
40659,ZVRA,Healthcare,Biotechnology,decil_3,2024,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.857353e+08,NaN,NaN
40660,ZWS,Industrials,Pollution & Treatment Controls,decil_5,2024,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.353906e+09,NaN,NaN


In [20]:
temp = pd.read_parquet("../../data/02_clean/df_metrics.parquet")
temp[temp["year"]=='2014']

,node,hrm,pozzi,degree,closeness,eig,year,beta,momentum
0,A,-0.113502,0.162029,2.219453,0.289924,0.000978,2014,1.289674,0.036170
1,AA,-0.242146,0.222550,1.851429,0.271225,0.000676,2014,1.470522,0.518100
2,AAL,-0.411204,0.213583,2.391266,0.237571,0.000902,2014,1.605365,1.117107
3,AAME,-0.566156,0.212748,0.223170,0.226859,0.000381,2014,0.045365,-0.002096
4,AAON,0.892336,0.186629,2.481132,0.331172,0.034447,2014,1.682752,0.066075
...,...,...,...,...,...,...,...,...,...
2718,ZNB,-0.473366,0.216124,0.299424,0.243640,0.000071,2014,0.628092,-0.102083
2719,ZROZ,-0.382266,0.214811,3.360004,0.238718,0.000283,2014,-0.692103,0.467750
2720,ZSL,-0.429236,0.276925,6.440218,0.211660,0.000012,2014,0.112141,0.300398
2721,ZTR,-0.098267,0.105097,0.509073,0.301451,0.001542,2014,0.451517,0.094877


In [ ]:
final_panel_df.to_csv("../../data/07_portfolios_metadata/complete_metadata.csv")

NameError: name 'final_panel_df' is not defined

In [5]:
import pandas as pd 

df = pd.read_parquet("../../data/07_portfolios_metadata/complete_metadata_hcm.parquet")
df

,Ticker,Sector,Industry,portfolio,year,beta_2014,momentum_2014,mcap_2015,beta_2015,momentum_2015,...,momentum_2021,mcap_2022,beta_2022,momentum_2022,mcap_2023,beta_2023,momentum_2023,mcap_2024,beta_2024,momentum_2024
0,A,Healthcare,Diagnostics & Research,decil_6,2014,1.289674,0.036170,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AA,Basic Materials,Aluminum,decil_7,2014,1.470522,0.518100,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AAL,Industrials,Airlines,decil_8,2014,1.605365,1.117107,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AAME,Financial,Insurance - Life,decil_9,2014,0.045365,-0.002096,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AAON,Industrials,Building Products & Equipment,decil_1,2014,1.682752,0.066075,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40657,ZTS,Healthcare,Drug Manufacturers - Specialty & Generic,decil_5,2024,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.292747e+10,0.623014,-0.166471
40658,ZUMZ,Consumer Cyclical,Apparel Retail,decil_1,2024,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.539357e+08,1.244622,-0.073231
40659,ZVRA,Healthcare,Biotechnology,decil_6,2024,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.857353e+08,NaN,NaN
40660,ZWS,Industrials,Pollution & Treatment Controls,decil_3,2024,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.353906e+09,NaN,NaN
